# Aim of this notebook
The goal of this note book is to automate the FAIR assesment of mutliple online resources through the [FAIR-Checker tool](https://fair-checker.france-bioinformatique.fr). 
All resulst are stored in a matrix and serialized into a CSV file. Scores can be interpreted as follows: 
 - 0 -> `failure`
 - 1 -> `weak` assesment
 - 2 -> `strong` assesment

To run this notebook you just need the `requests` and `pandas` python libraries. 

The FAIR-Checker API is better described at https://fair-checker.france-bioinformatique.fr/swagger 

Please report any issue at https://github.com/IFB-ElixirFr/fair-checker/issues or contact alban.gaignard@univ-nantes.fr. 

In [1]:
import time
import requests
import pandas as pd
from rdflib import ConjunctiveGraph
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import matplotlib as mpl

In [2]:
apis = ["https://tess.elixir-europe.org/materials.json_api",
        "https://tess.elixir-europe.org/events.json_api",
        "https://tess.elixir-europe.org/content_providers.json_api",
        "https://tess.elixir-europe.org/workflows.json_api"]

event_urls = []
data = requests.get('https://tess.elixir-europe.org/events.json_api')
print(len(data.json()["data"]))
for e in tqdm(data.json()["data"]):
    u = e["attributes"]["url"]
    event_urls.append(u)
    #print(u)
print(event_urls)


10


  0%|          | 0/10 [00:00<?, ?it/s]

['https://www.biocommons.org.au/events/protein-binder-series', 'https://datenkompetenz.cloud/en/ask-our-experts/', 'https://training.vib.be/all-trainings/career-guidance-phds-and-postdocs-8', 'https://training.vib.be/all-trainings/career-guidance-phds-and-postdocs-7', 'https://training.vib.be/all-trainings/how-manage-your-phd-9', 'https://training.vib.be/all-trainings/reproducible-data-analysis-0', 'https://www.ebi.ac.uk/training/events/concepts-methods-and-resources-pangenomics', 'https://www.ebi.ac.uk/training/events/uniprot-2025-26', 'https://training.vib.be/all-trainings/show-dont-tell-creating-visuals-about-your-data-0', 'https://training.vib.be/all-trainings/presenting-stage-3']


## Input dataset

In [28]:
bench_urls = [
    "https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/JGO6VI",
    "https://www.data.gouv.fr/en/datasets/donnees-relatives-a-lepidemie-de-covid-19-en-france-vue-densemble/",
    "https://www.kaggle.com/datasets/imdevskp/corona-virus-report",
    "https://data.who.int/dashboards/covid19/data",
    "https://data.opendatasoft.com/explore/dataset/donnees-hospitalieres-covid-19-dep-france%40public/table/?disjunctive.countrycode_iso_3166_1_alpha3&disjunctive.nom_dep_min",

    "https://bio.tools/bwa",

    "https://hpo.jax.org",
    "https://www.ebi.ac.uk/ols4/ontologies/go",

    "https://tess.elixir-europe.org/materials/make-your-research-fairer-with-quarto-github-and-zenodo",
    "https://moodle.polytechnique.fr/course/index.php?categoryid=1018",

    "http://doi.org/10.1594/PANGAEA.908011",
    "http://www.kaggle.com/allen-institute-for-ai/CORD-19-research-challenge",
    "https://data.rivm.nl/meta/srv/eng/rdf.metadata.get?uuid=1c0fcd57-1102-4620-9cfa-441e93ea5604&approved=true"
]

In [4]:
FC_all_metrics_url = "https://fair-checker.france-bioinformatique.fr/api/check/metrics_all"


#res = requests.get(url=FC_all_metrics_url, params={"url": "https://data.who.int/dashboards/covid19/data"})
#evaluations = res.json()
#evaluations

def retrieve_rdf(url):
    FC_get_md = "https://fair-checker.france-bioinformatique.fr/api/inspect/get_rdf_metadata"
    kg = ConjunctiveGraph()
    res = requests.get(url=FC_get_md, params={"url": url})
    try:
        kg.parse(data=res.text, format="json-ld")
    except Exception as e:
        print(e)
    print(f"Loaded {len(kg)} RDF triples from {url}")
    return kg

In [5]:
dump = ConjunctiveGraph()
for event in tqdm(event_urls):
    kg = retrieve_rdf(event)
    dump += kg

print(len(dump))

/var/folders/x1/d1nyvpvs0td0htxzsv9x6ggw0000gn/T/ipykernel_53739/2714016929.py:1: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  dump = ConjunctiveGraph()


  0%|          | 0/10 [00:00<?, ?it/s]

/var/folders/x1/d1nyvpvs0td0htxzsv9x6ggw0000gn/T/ipykernel_53739/2594897655.py:9: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  kg = ConjunctiveGraph()


Loaded 45 RDF triples from https://www.biocommons.org.au/events/protein-binder-series
Loaded 0 RDF triples from https://datenkompetenz.cloud/en/ask-our-experts/
Loaded 68 RDF triples from https://training.vib.be/all-trainings/career-guidance-phds-and-postdocs-8
Loaded 68 RDF triples from https://training.vib.be/all-trainings/career-guidance-phds-and-postdocs-7
Loaded 68 RDF triples from https://training.vib.be/all-trainings/how-manage-your-phd-9
Loaded 102 RDF triples from https://training.vib.be/all-trainings/reproducible-data-analysis-0
Loaded 4 RDF triples from https://www.ebi.ac.uk/training/events/concepts-methods-and-resources-pangenomics
Loaded 4 RDF triples from https://www.ebi.ac.uk/training/events/uniprot-2025-26
Loaded 62 RDF triples from https://training.vib.be/all-trainings/show-dont-tell-creating-visuals-about-your-data-0
Loaded 69 RDF triples from https://training.vib.be/all-trainings/presenting-stage-3
479


## FAIR assesment over all inputs 

In [31]:
from json import JSONDecodeError

df = pd.DataFrame()
rows = []
kg = ConjunctiveGraph()

for u in tqdm(bench_urls):
    # call to the FC API
    start = time.time()
    res = requests.get(url=FC_all_metrics_url, params={"url": u})
    eval_in_sec = time.time() - start

    try:
        evaluations = res.json()
        row = {"URL": u}

        # iterating over all evaluation results
        for e in evaluations:
            row[e["metric"]] = int(e["score"])
        # row["duration (s)"] = round(eval_in_sec, 2)
        #    print(row)
        rows.append(row)
    except JSONDecodeError as e:
        pass

    kg += retrieve_rdf(u)

kg.serialize("out.ttl", format="turtle")

  0%|          | 0/13 [00:00<?, ?it/s]

Loaded 136 RDF triples from https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/JGO6VI
Loaded 198 RDF triples from https://www.data.gouv.fr/en/datasets/donnees-relatives-a-lepidemie-de-covid-19-en-france-vue-densemble/
Loaded 110 RDF triples from https://www.kaggle.com/datasets/imdevskp/corona-virus-report
Loaded 6 RDF triples from https://data.who.int/dashboards/covid19/data
Loaded 7 RDF triples from https://data.opendatasoft.com/explore/dataset/donnees-hospitalieres-covid-19-dep-france%40public/table/?disjunctive.countrycode_iso_3166_1_alpha3&disjunctive.nom_dep_min
Loaded 127 RDF triples from https://bio.tools/bwa
Loaded 0 RDF triples from https://hpo.jax.org
Loaded 0 RDF triples from https://www.ebi.ac.uk/ols4/ontologies/go
Loaded 27 RDF triples from https://tess.elixir-europe.org/materials/make-your-research-fairer-with-quarto-github-and-zenodo
Loaded 0 RDF triples from https://moodle.polytechnique.fr/course/index.php?categoryid=1018
Loaded 168 RDF triples fro

<Graph identifier=N41091bfe848340c2aa9bed03b6bcf182 (<class 'rdflib.graph.ConjunctiveGraph'>)>

## Evaluation matrix

In [32]:
from IPython.display import display, Markdown

df = pd.DataFrame.from_records(rows)
md = df.to_markdown()
display(Markdown(md))

|    | URL                                                                                                                                                                       |   F1A |   F1B |   F2A |   F2B |   A1.1 |   A1.2 |   I1 |   I2 |   I3 |   R1.1 |   R1.2 |   R1.3 |
|---:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------:|------:|------:|------:|-------:|-------:|-----:|-----:|-----:|-------:|-------:|-------:|
|  0 | https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/JGO6VI                                                                                           |     2 |     2 |     1 |     1 |      2 |      2 |    1 |    1 |    2 |      2 |      2 |      1 |
|  1 | https://www.data.gouv.fr/en/datasets/donnees-relatives-a-lepidemie-de-covid-19-en-france-vue-densemble/                                                                   |     2 |     0 |     1 |     1 |      2 |      2 |    1 |    1 |    2 |      2 |      2 |      1 |
|  2 | https://www.kaggle.com/datasets/imdevskp/corona-virus-report                                                                                                              |     2 |     2 |     1 |     1 |      2 |      2 |    1 |    1 |    0 |      2 |      2 |      1 |
|  3 | https://data.who.int/dashboards/covid19/data                                                                                                                              |     2 |     0 |     1 |     2 |      2 |      0 |    1 |    2 |    0 |      0 |      0 |      2 |
|  4 | https://data.opendatasoft.com/explore/dataset/donnees-hospitalieres-covid-19-dep-france%40public/table/?disjunctive.countrycode_iso_3166_1_alpha3&disjunctive.nom_dep_min |     2 |     0 |     1 |     2 |      2 |      0 |    1 |    2 |    0 |      0 |      0 |      2 |
|  5 | https://bio.tools/bwa                                                                                                                                                     |     2 |     2 |     1 |     1 |      2 |      0 |    1 |    1 |    2 |      0 |      0 |      1 |
|  6 | https://hpo.jax.org                                                                                                                                                       |     2 |     0 |     0 |     0 |      2 |      0 |    0 |    0 |    0 |      0 |      0 |      0 |
|  7 | https://www.ebi.ac.uk/ols4/ontologies/go                                                                                                                                  |     2 |     0 |     0 |     0 |      2 |      0 |    0 |    0 |    0 |      0 |      0 |      0 |
|  8 | https://tess.elixir-europe.org/materials/make-your-research-fairer-with-quarto-github-and-zenodo                                                                          |     2 |     0 |     1 |     1 |      2 |      2 |    1 |    1 |    2 |      2 |      2 |      1 |
|  9 | https://moodle.polytechnique.fr/course/index.php?categoryid=1018                                                                                                          |     2 |     0 |     0 |     0 |      2 |      0 |    0 |    0 |    0 |      0 |      0 |      0 |
| 10 | http://doi.org/10.1594/PANGAEA.908011                                                                                                                                     |     2 |     2 |     1 |     2 |      2 |      2 |    1 |    2 |    2 |      2 |      2 |      2 |
| 11 | http://www.kaggle.com/allen-institute-for-ai/CORD-19-research-challenge                                                                                                   |     2 |     2 |     1 |     1 |      2 |      2 |    1 |    1 |    0 |      2 |      2 |      1 |
| 12 | https://data.rivm.nl/meta/srv/eng/rdf.metadata.get?uuid=1c0fcd57-1102-4620-9cfa-441e93ea5604&approved=true                                                                |     2 |     2 |     2 |     1 |      2 |      2 |    2 |    1 |    2 |      2 |      2 |      1 |

In [33]:
# import seaborn as sns 
# sns.histplot(data=df["R1.3"])
# plt.savefig('r13.png', dpi=300)
df

,URL,F1A,F1B,F2A,F2B,A1.1,A1.2,I1,I2,I3,R1.1,R1.2,R1.3
0,https://dataverse.harvard.edu/dataset.xhtml?pe...,2,2,1,1,2,2,1,1,2,2,2,1
1,https://www.data.gouv.fr/en/datasets/donnees-r...,2,0,1,1,2,2,1,1,2,2,2,1
2,https://www.kaggle.com/datasets/imdevskp/coron...,2,2,1,1,2,2,1,1,0,2,2,1
3,https://data.who.int/dashboards/covid19/data,2,0,1,2,2,0,1,2,0,0,0,2
4,https://data.opendatasoft.com/explore/dataset/...,2,0,1,2,2,0,1,2,0,0,0,2
5,https://bio.tools/bwa,2,2,1,1,2,0,1,1,2,0,0,1
6,https://hpo.jax.org,2,0,0,0,2,0,0,0,0,0,0,0
7,https://www.ebi.ac.uk/ols4/ontologies/go,2,0,0,0,2,0,0,0,0,0,0,0
8,https://tess.elixir-europe.org/materials/make-...,2,0,1,1,2,2,1,1,2,2,2,1
9,https://moodle.polytechnique.fr/course/index.p...,2,0,0,0,2,0,0,0,0,0,0,0


In [34]:
df['F_score'] = df.apply(lambda row: round((row["F1A"] + row["F1B"] + row["F2A"] + row["F2B"]) * 100 / 8, 1), axis=1)
df['A_score'] = df.apply(lambda row: round((row["A1.1"] + row["A1.2"]) * 100 / 4, 1), axis=1)
df['I_score'] = df.apply(lambda row: round((row["I1"] + row["I2"] + row["I3"]) * 100 / 6, 1), axis=1)
df['R_score'] = df.apply(lambda row: round((row["R1.1"] + row["R1.2"] + row["R1.3"]) * 100 / 6, 1), axis=1)
#df['FAIR_score'] = df.apply(lambda row: round((row["F_score"] + row["A_score"] + row["I_score"] + row["R_score"])/4, 1), axis = 1)
df['FAIR_score'] = df.apply(lambda row: round((row["F1A"] + row["F1B"] + row["F2A"] + row["F2B"]
                                               + row["A1.1"] + row["A1.2"]
                                               + row["I1"] + row["I2"] + row["I3"]
                                               + row["R1.1"] + row["R1.2"] + row["R1.3"]) * 100 / 24, 1), axis=1)

df.to_csv("fairchecker_dekalog_evals.csv")
df

,URL,F1A,F1B,F2A,F2B,A1.1,A1.2,I1,I2,I3,R1.1,R1.2,R1.3,F_score,A_score,I_score,R_score,FAIR_score
0,https://dataverse.harvard.edu/dataset.xhtml?pe...,2,2,1,1,2,2,1,1,2,2,2,1,75.0,100.0,66.7,83.3,79.2
1,https://www.data.gouv.fr/en/datasets/donnees-r...,2,0,1,1,2,2,1,1,2,2,2,1,50.0,100.0,66.7,83.3,70.8
2,https://www.kaggle.com/datasets/imdevskp/coron...,2,2,1,1,2,2,1,1,0,2,2,1,75.0,100.0,33.3,83.3,70.8
3,https://data.who.int/dashboards/covid19/data,2,0,1,2,2,0,1,2,0,0,0,2,62.5,50.0,50.0,33.3,50.0
4,https://data.opendatasoft.com/explore/dataset/...,2,0,1,2,2,0,1,2,0,0,0,2,62.5,50.0,50.0,33.3,50.0
5,https://bio.tools/bwa,2,2,1,1,2,0,1,1,2,0,0,1,75.0,50.0,66.7,16.7,54.2
6,https://hpo.jax.org,2,0,0,0,2,0,0,0,0,0,0,0,25.0,50.0,0.0,0.0,16.7
7,https://www.ebi.ac.uk/ols4/ontologies/go,2,0,0,0,2,0,0,0,0,0,0,0,25.0,50.0,0.0,0.0,16.7
8,https://tess.elixir-europe.org/materials/make-...,2,0,1,1,2,2,1,1,2,2,2,1,50.0,100.0,66.7,83.3,70.8
9,https://moodle.polytechnique.fr/course/index.p...,2,0,0,0,2,0,0,0,0,0,0,0,25.0,50.0,0.0,0.0,16.7


## Visualisation

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

#df = pd.read_csv("fc_dekalog_evals.csv")
my_plot = sns.barplot(x='URL', y='FAIR_score', data=df)
my_plot.set_xticklabels(my_plot.get_xticklabels(), rotation=90)

my_plot

# Manual evaluations with F-UJI

In [48]:
fuji_evals = {}
fuji_evals["harvard_dataverse"] = 75
fuji_evals["data.gouv.fr"] = 52
fuji_evals["kaggle"] = 60
fuji_evals["who"] = 27
fuji_evals["opendatasoft"] = 31
fuji_evals["bio.tools"] = 18
fuji_evals["HPO"] = 18
fuji_evals["HPO_ols"] = 18
fuji_evals["tess"] = 39
fuji_evals["moodle"] = 4
fuji_evals["pangaea"] = 91
fuji_evals["kaggle_2"] = 60
fuji_evals["rdf_content_neg"] = 43

In [26]:
import glob, json
import pandas as pd

rows = []


def parse_fuji_metrics(data, f):
    u = f.split("/")[-1]
    row = {"FILE": u}
    for result in data["results"]:
        print(result["metric_identifier"],
              result["score"]["earned"],
              result["score"]["total"])
        row[result["metric_identifier"]] = result["score"]["earned"]
    return row


def parse_fuji_summary(data, f):
    u = f.split("/")[-1]
    row = {"FILE": u}
    s = data["summary"]["score_earned"]
    for key, value in s.items():
        row[key] = value
    return row


for f in glob.glob("./dekalog_tests/*.json"):
    print(f)
    with open(f) as file:
        s = str("".join(file.readlines()))
        data = json.loads(s)
        row = parse_fuji_summary(data, f)
        #row = parse_fuji_metrics(data, f)
        rows.append(row)

df = pd.DataFrame.from_records(rows)
df.to_csv("fuji_dekalog_evals_summary.csv")
#df.to_csv("fuji_dekalog_evals_metrics.csv")
df

./dekalog_tests/https___www.kaggle.com_datasets_imdevskp_corona-virus-report.json
FsF-F1-01D 1 1
FsF-F1-02D 0 1
FsF-F2-01M 0.5 2
FsF-F3-01M 1 1
FsF-F4-01M 1 2
FsF-I1-01M 1 2
FsF-I2-01M 1 1
FsF-I3-01M 1 1
FsF-R1-01MD 2 4
FsF-R1.1-01M 1 2
FsF-A1-01M 1 1
FsF-R1.2-01M 1 2
FsF-R1.3-01M 1 1
FsF-R1.3-02D 0 1
FsF-A1-03D 1 1
FsF-A1-02M 1 1
./dekalog_tests/https___data.opendatasoft.com_explore_dataset_donnees-hospitalieres-covid-19-dep-france@public_table_?disjunctive.countrycode_iso_3166_1_alpha3&disjunctive.nom_dep_min.json
FsF-F1-01D 1 1
FsF-F1-02D 0 1
FsF-F2-01M 0.5 2
FsF-F3-01M 0 1
FsF-F4-01M 1 2
FsF-I1-01M 0 2
FsF-I2-01M 0 1
FsF-I3-01M 0 1
FsF-R1-01MD 1 4
FsF-R1.1-01M 1 2
FsF-A1-01M 0 1
FsF-R1.2-01M 1 2
FsF-R1.3-01M 1 1
FsF-R1.3-02D 0 1
FsF-A1-03D 0 1
FsF-A1-02M 1 1
./dekalog_tests/https___data.who.int_dashboards_covid19_data.json
FsF-F1-01D 1 1
FsF-F1-02D 0 1
FsF-F2-01M 0.5 2
FsF-F3-01M 0 1
FsF-F4-01M 1 2
FsF-I1-01M 0 2
FsF-I2-01M 0 1
FsF-I3-01M 0 1
FsF-R1-01MD 1 4
FsF-R1.1-01M 0 2
FsF-A1

,FILE,FsF-F1-01D,FsF-F1-02D,FsF-F2-01M,FsF-F3-01M,FsF-F4-01M,FsF-I1-01M,FsF-I2-01M,FsF-I3-01M,FsF-R1-01MD,FsF-R1.1-01M,FsF-A1-01M,FsF-R1.2-01M,FsF-R1.3-01M,FsF-R1.3-02D,FsF-A1-03D,FsF-A1-02M
0,https___www.kaggle.com_datasets_imdevskp_coron...,1,0,0.5,1,1,1,1,1,2,1,1,1,1,0,1,1
1,https___data.opendatasoft.com_explore_dataset_...,1,0,0.5,0,1,0,0,0,1,1,0,1,1,0,0,1
2,https___data.who.int_dashboards_covid19_data.json,1,0,0.5,0,1,0,0,0,1,0,0,1,1,0,0,1
3,http___doi.org_10.1594_PANGAEA.908011.json,1,1,2.0,1,2,2,1,1,3,2,1,1,1,1,1,1
4,https___data.rivm.nl_meta_srv_eng_rdf.metadata...,1,0,0.5,1,0,1,1,0,1,0,0,1,1,1,1,1
5,https___hpo.jax.org.json,1,0,0.5,0,1,0,0,0,0,0,0,0,1,0,0,1
6,https___moodle.polytechnique.fr_course_index.p...,1,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,https___dataverse.harvard.edu_dataset.xhtml?pe...,1,1,2.0,1,2,2,0,1,3,0,0,1,1,1,1,1
8,http___www.kaggle.com_allen-institute-for-ai_C...,1,0,0.5,1,1,1,1,1,2,1,1,1,1,0,1,1
9,https___tess.elixir-europe.org_materials_make-...,1,0,0.5,0,1,1,1,0,1,1,0,1,1,0,0,1
